In [ ]:
# ============================================================
# Maritime Logistics Operations Analysis
# Ondatax DataDNA Challenge — March 2026
# ============================================================
# Script: 03_targeted_analysis.py
# Purpose: Load raw 4-table star schema, perform data quality
#          checks, remove duplicates, fix data types, enrich
#          with calculated fields, merge into analysis-ready dataset
# Input:   data/raw/*.csv
# Output:  data/processed/q3_master_table.csv
# ============================================================

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

df=pd.read_csv('maritime_analysis_ready.csv')
df['date_id']=pd.to_datetime(df['date_id'])

pre_suez_baseline=447.94
recovery_threshold=0.10

In [3]:
print("="*60)
print("Q1 - SUEZ EFFECT")
print("="*60)

# ── Q1a: Recovery time ───────────────────────────────────────
# How many months until avg duration returned within 10%
# of pre-Suez baseline (447.94 hrs)?
# Recovery threshold = baseline × 1.10 = 492.73 hrs

recovery_limit=pre_suez_baseline*(1+recovery_threshold)

monthly=df.set_index('date_id').resample('ME').agg(
    avg_duration=('move_duration','mean'),
    total_movements=('movement_id','count')
).round(2)

post_disruption=monthly[monthly.index>'2021-06-01'].copy()
post_disruption['recovered']=post_disruption['avg_duration']<=recovery_limit

recoverd_months=post_disruption[post_disruption['recovered']]

print("----Q1a: Recovery time----")
print(f"Pre suez baseline: {pre_suez_baseline:.2f} hrs")
print(f"Recovery threshold: {recovery_limit:.2f} hrs (within 10% of baseline)")

if len(recoverd_months)>0:
    first_recovery=recoverd_months.index[0]
    months_to_recover=len(post_disruption[post_disruption.index<=first_recovery])
    print(f"First recovered month: {first_recovery.strftime('%B %Y')}")
    print(f"Months to recover: {months_to_recover} months after disruption")
else:
    print("Operations never recovered to within 10% of pre suez baseline")
    print(f"Closest post-suez month: {post_disruption['avg_duration'].min():.2f} hrs")
    print(f"Still {post_disruption['avg_duration'].min()-pre_suez_baseline:.2f} hrs above baseline")

print("Post disruption monthly trend (first 12 months): ")
print(post_disruption[['avg_duration','recovered']].head(12))

Q1 - SUEZ EFFECT
----Q1a: Recovery time----
Pre suez baseline: 447.94 hrs
Recovery threshold: 492.73 hrs (within 10% of baseline)
First recovered month: June 2021
Months to recover: 1 months after disruption
Post disruption monthly trend (first 12 months): 
            avg_duration  recovered
date_id                            
2021-06-30        476.11       True
2021-07-31        430.91       True
2021-08-31        525.35      False
2021-09-30        473.44       True
2021-10-31        492.07       True
2021-11-30        540.45      False
2021-12-31        539.17      False
2022-01-31        301.78       True
2022-02-28        487.65       True
2022-03-31        643.11      False
2022-04-30        478.40       True
2022-05-31        504.14      False


In [4]:
print("---Q1b: Regional recovery comparison----")
regional_period=df.groupby(['suez_period','regional_hub']).agg(
    avg_duration=('move_duration','mean'),
    movements=('movement_id','count')
).round(2).reset_index()

pivot=regional_period.pivot(
    index='regional_hub',
    columns='suez_period',
    values='avg_duration')[['pre_suez','suez_disruption','post_suez']]

pivot['spike_pct']=((pivot['suez_disruption']-pivot['pre_suez'])/pivot['pre_suez']*100).round(1)
pivot['recovery_pct']=((pivot['post_suez']-pivot['pre_suez'])/pivot['pre_suez']*100).round(1)
pivot['fully_recoverd']=pivot['recovery_pct']<=10

print(pivot.sort_values('spike_pct',ascending=False))
print(f"Region fully recovered (within 10% of baseline):")
print(f"{pivot['fully_recoverd'].sum()} of {len(pivot)}")

---Q1b: Regional recovery comparison----
suez_period   pre_suez  suez_disruption  post_suez  spike_pct  recovery_pct  \
regional_hub                                                                  
LATAM           403.61           562.72     494.79       39.4          22.6   
EMEA            389.11           512.31     499.47       31.7          28.4   
APAC            494.33           530.76     509.06        7.4           3.0   
AMER            651.44           660.51     487.74        1.4         -25.1   

suez_period   fully_recoverd  
regional_hub                  
LATAM                  False  
EMEA                   False  
APAC                    True  
AMER                    True  
Region fully recovered (within 10% of baseline):
2 of 4


In [6]:
# ── Q2a: Correlation — utilization vs duration ───────────────
# Does being busier actually make a terminal slower?

print("="*60)
print("Q2 - INFRASTRUCTURE BOTTLENECK")
print("="*60)

terminal_stats=df.groupby(['terminal_id','terminal_name','regional_hub']).agg(
    total_movements=('movement_id','count'),
    avg_duration=('move_duration','mean'),
    total_containters=('container_count','sum')
).round(2).reset_index()

median_load=terminal_stats['total_movements'].median()
terminal_stats['utilization']=(terminal_stats['total_movements']/median_load).round(2)

correlation,p_value=stats.pearsonr(
    terminal_stats['utilization'],
    terminal_stats['avg_duration']
)

print("Q2a: Correlation - utilization vs avg duration")
print(f"Pearson correlation: {correlation:.3f}")
print(f"P-value: {p_value:.3f}")
print(f"Sample size: {len(terminal_stats)} terminals")

if abs(correlation)>=0.7:
    strength='strong'
elif abs(correlation)>=0.4:
    strength='moderate'
elif abs(correlation)>=0.1:
    strength='weak'
else:
    strength='negligible'

direction='positive' if correlation>0 else 'negative'
significant='statistically significant' if p_value<0.05 else 'not statistically significant'

print(f"Interpretation: {strength} {direction} correlation, {significant}")
print(f"Meaning: {'Busier terminal tend to have longer duration' if correlation>0 else 'Busier terminal tend to be faster'}")

Q2 - INFRASTRUCTURE BOTTLENECK
Q2a: Correlation - utilization vs avg duration
Pearson correlatio: -0.044
P-value: 0.761
Sample size: 50 terminals
Interpretation: negligible negative correlation, not statistically significant
Meaning: Busier terminal tend to be faster


In [10]:
# ── Q2b: Vessel category mix at congested terminals ──────────
# Which vessel types are dominating the high-load terminals?

print("----Q2b: Vessel category mix at top 10 busiest terminals----")
top10_ids=terminal_stats.nlargest(10, 'total_movements')['terminal_id'].tolist()

vessel_mix=df[df['terminal_id'].isin(top10_ids)].groupby(
    ['terminal_name','vessel_category_x']
).agg(
    movements=('movement_id','count'),
    avg_duration=('move_duration','mean')
).round(2).reset_index()

vessel_pivot=vessel_mix.pivot_table(
    index='terminal_name',
    columns='vessel_category_x',
    values='movements',
    fill_value=0
)
print(vessel_pivot)

print("\nAvg duration per vessel category at top 10 terminals:")
duration_pivot=vessel_mix.pivot_table(
    index='terminal_name',
    columns='vessel_category_x',
    values='avg_duration',
    fill_value=0
).round(1)
print(duration_pivot)

----Q2b: Vessel category mix at top 10 busiest terminals----
vessel_category_x  Cargo  Container  Passenger  Tanker
terminal_name                                         
David Dixon         11.0       15.0       15.0    11.0
Deborah Morgan       7.0       12.0       12.0     9.0
Elizabeth Norman    13.0        6.0        7.0    16.0
Isaac Moore         14.0       10.0        7.0    14.0
Jamie Banks         15.0       10.0       10.0     8.0
Lisa Carter          5.0       15.0       11.0    11.0
Nicole Miller       11.0        6.0        9.0    15.0
Shannon Gray        15.0        5.0       23.0    15.0
Tammy Strong         8.0       16.0        8.0    15.0
Tara Walker          7.0        9.0       10.0    19.0

Avg duration per vessel category at top 10 terminals:
vessel_category_x  Cargo  Container  Passenger  Tanker
terminal_name                                         
David Dixon        538.5      289.6      515.6   375.2
Deborah Morgan     496.8      531.5      552.2   418.4
Eliz

In [14]:
# ── Q2c: Capacity expansion candidates ───────────────────────
# Formally rank terminals by expansion need
# Score = utilization × avg_duration (high load + slow = most urgent)


print("----Q2c: Capacity expansion candidates----")
terminal_stats['expansion_score']=(
    terminal_stats['utilization']*terminal_stats['avg_duration']
).round(2)


expansion_candidates=terminal_stats.nlargest(10,'expansion_score')[
    ['terminal_name','regional_hub','total_movements','avg_duration','utilization','expansion_score']
]
print(expansion_candidates)

print("\nUnderutilized terminals (spare capacity available)")
underutilized=terminal_stats[terminal_stats['utilization']<0.8][[
    'terminal_name','regional_hub','total_movements','avg_duration','utilization'
]].sort_values('utilization')

print(underutilized)

----Q2c: Capacity expansion candidates----
       terminal_name regional_hub  total_movements  avg_duration  utilization  \
42      Shannon Gray         APAC               58        531.42         1.87   
8       Tammy Strong         EMEA               47        606.60         1.52   
11       Lisa Carter        LATAM               42        596.88         1.35   
9        Jamie Banks         APAC               43        543.70         1.39   
41       David Dixon        LATAM               52        425.56         1.68   
26       Tara Walker         APAC               45        479.64         1.45   
24  Elizabeth Norman         AMER               42        512.08         1.35   
18       Isaac Moore         EMEA               45        474.53         1.45   
46      John Bridges         AMER               34        607.49         1.10   
29     Nicole Miller         EMEA               41        497.30         1.32   

    expansion_score  
42           993.76  
8            922.03  

In [22]:
# ============================================================
# Q3 — EFFICIENCY ANOMALIES
# ============================================================
 
print("\n" + "=" * 60)
print("Q3 — EFFICIENCY ANOMALIES")
print("=" * 60)
 
# ── Q3a: Factor ranking — what predicts duration most? ───────
# Compare how much each factor reduces variance in duration
# Method: calculate variance explained by each grouping factor
# Higher variance between groups = stronger predictor

print("----Q3a: Which factor most predicts move duration?----")

factors={
    'vessel_category':df['vessel_category_x'],
    'regional_hub':df['regional_hub'],
    'shift':df['shift']
}

total_variance=df['move_duration'].var()
results=[]

for factor_name,factor_col in factors.items():
    group_means=df.groupby(factor_col)['move_duration'].mean()
    group_counts=df.groupby(factor_col)['move_duration'].count()

    overall_mean=df['move_duration'].mean()
    between_var=sum(
        group_counts[g]*(group_means[g]-overall_mean)**2
        for g in group_means.index
    )/len(df)

    variance_explained_pct=(between_var/total_variance*100).round(2)
    results.append({
        'factor':factor_name,
        'variance_explained_pct':variance_explained_pct,
        'group_count':len(group_means),
        'highest_avg_duration':round(group_means.max(), 1),
        'lowest_avg_duration':round(group_means.min(), 1),
        'gap_hrs':round(group_means.max()-group_means.min(), 1)
    })

clean_df=df[df['vessel_age'].notna()]
age_corr,age_p=stats.pearsonr(clean_df['vessel_age'],clean_df['move_duration'])
results.append({
    'factor':'vessel_age',
    'variance_explained_pct': round(age_corr ** 2 * 100, 2),
    'group_count':'continuous',
    'highest_avg_duration':'-',
    'lowest_avg_duration':'-',
    'gap_hrs':f'corr={age_corr:.3f}'
})

factor_df=pd.DataFrame(results).sort_values('variance_explained_pct',ascending=False)

print(factor_df.to_string(index=False))
print(f"\nMost powerful predictor: {factor_df.iloc[0]['factor']} "
f"({factor_df.iloc[0]['variance_explained_pct']}% of variance explained)")


Q3 — EFFICIENCY ANOMALIES
----Q3a: Which factor most predicts move duration?----
         factor  variance_explained_pct group_count highest_avg_duration lowest_avg_duration    gap_hrs
vessel_category                    0.55           4                534.6               473.2       61.4
          shift                    0.05           2                506.0               493.2       12.8
   regional_hub                    0.04           4                509.2               494.2       15.0
     vessel_age                    0.00  continuous                    -                   - corr=0.001

Most powerful predictor: vessel_category (0.55% of variance explained)


In [28]:
print("\n── Q3b: Outlier terminals (duration > 1.5 std above regional avg) ──")
regional_stats = df.groupby('regional_hub')['move_duration'].agg(
    regional_mean = 'mean',
    regional_std  = 'std'
).round(2)
 
terminal_regional = terminal_stats.merge(
    regional_stats, on='regional_hub', how='left'
)
terminal_regional['pct_above_regional'] = (
    (terminal_regional['avg_duration'] - terminal_regional['regional_mean'])
    / terminal_regional['regional_mean'] * 100
).round(1)
 
outliers = terminal_regional[
    terminal_regional['pct_above_regional'] > 20
][['terminal_name', 'regional_hub', 'avg_duration',
   'regional_mean', 'pct_above_regional']].sort_values(
    'pct_above_regional', ascending=False
)
print(outliers)


── Q3b: Outlier terminals (duration > 1.5 std above regional avg) ──
      terminal_name regional_hub  avg_duration  regional_mean  \
12    Lauren Little        LATAM        624.29         494.19   
8      Tammy Strong         EMEA        606.60         495.18   
43  Michael Stewart         EMEA        602.22         495.18   
46     John Bridges         AMER        607.49         501.38   
11      Lisa Carter        LATAM        596.88         494.19   
15    John Anderson         EMEA        595.58         495.18   

    pct_above_regional  
12                26.3  
8                 22.5  
43                21.6  
46                21.2  
11                20.8  
15                20.3  


In [ ]:
# ── Q3c: Shift analysis — formal test ────────────────────────
# Is the day vs night duration difference statistically significant?
 
print("\n── Q3c: Shift analysis — is the difference real? ──")

day_duration=df[df['shift']=='Day']['move_duration']
night_duration=df[df['shift']=='Night']['move_duration']

t_stat,p_val=stats.ttest_ind(day_duration,night_duration)

print(f"Day shift avg: {day_duration.mean():.2f} hrs(n={len(day_duration)})")
print(f"Night shift avg: {night_duration.mean():.2f} hrs(n={len(night_duration)})")
print(f"Difference: {day_duration.mean()-night_duration.mean():.2f} hrs")
print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_val:.3f}")
print(f"Conclusion: {'Difference is statistically significant' if p_val<0.05 else 'Difference is NOT statistically significant - shift has no real effect on duration'}")   


── Q3c: Shift analysis — is the difference real? ──
Day shift avg: 505.98 hrs(n=793)
Night shift avg: 493.19 hrs(n=808)
Difference: 12.80 hrs
T-statistic: 0.882
P-value: 0.378
Conclusion: Different is NOT statistically significant - shift has no real affect on duration


In [32]:
import pandas as pd
fact = pd.read_csv('fact_cargo_movements.csv')

# How many truly unique movement_ids exist?
print(fact['movement_id'].nunique())

# How many times does each movement_id repeat on average?
print(fact.groupby('movement_id').size().describe())

# Show the most repeated movement_ids
print(fact.groupby('movement_id').size().sort_values(ascending=False).head(10))

1001
count    1001.000000
mean       14.985015
std         3.807726
min         5.000000
25%        12.000000
50%        15.000000
75%        18.000000
max        26.000000
dtype: float64
movement_id
659    26
991    26
528    26
255    26
339    26
124    26
960    25
681    25
299    25
882    24
dtype: int64
